# 02 · NER + NEL + embed, in parallel

**Runs on: the GPU box.** No `kubectl`, no Elasticsearch, no Neo4j — every
store in this notebook is in memory. Its only network dependency is S3.

Downloads the corpus shards, runs the library's integrated annotate pass
(GLiNER → HNSW linker → bge-base) across N processes, and uploads one annotated
parquet per shard. The parquet carries mentions, entity links and the
embeddings, so the store box never recomputes any of it.

```
  STORE BOX (kubectl)              S3  s3.wisefood-project.eu           GPU BOX
  ─────────────────────            ──────────────────────────           ───────
  01_corpus                                                             02_annotate
    articles ──chunk──────────────> corpus/shard_*.csv  ──────────────>  download
                                    ontology/foodon.*   ──────────────>  download
                                                                         NER + NEL + embed
                                                                         (parallel)
  03_layers                         annotated/shard_*.parquet  <───────  upload
    download <──────────────────────
    ingest → prod Elasticsearch
    Layer A/B/C → prod Neo4j
```

Neither box needs what the other has. The GPU box never reaches Elasticsearch
or Neo4j — they are `ClusterIP` with no ingress and it could not if it wanted
to. The store box never needs a GPU. MinIO is the only thing both can see, and
it already has one: `s3.wisefood-project.eu` resolves publicly and answers
`/minio/health/live`.

`FS_RUN_ID` is the join. Use the same string on both machines; it namespaces
every object under `runs/<FS_RUN_ID>/`, so two builds cannot interleave.

## 0. Preflight

```bash
export S3_ACCESS_KEY=root
export S3_SECRET_KEY=...
export FS_RUN_ID=2026-09-21-full   # the SAME string as on the store box

export FS_WORKERS=4                # processes; 3-4 GB RAM each
export FS_WORKER_THREADS=2         # torch threads inside each
export FS_ANNOTATE_BATCH=64        # raise this on a GPU, not FS_WORKERS
```

**Sizing.** Each worker holds GLiNER (~1.5 GB), bge-base (~0.4 GB), the HNSW
index and the ontology. On a GPU, one or two workers saturate the card and more
only add contention — raise the batch instead. On CPU, workers are the lever.

Needs `foodscholar[annotate,chunking]`.

In [ ]:
import os, sys, time, json
from pathlib import Path

REQUIRED = ["S3_ACCESS_KEY", "S3_SECRET_KEY", "FS_RUN_ID"]
missing = [v for v in REQUIRED if not os.environ.get(v)]
if missing:
    raise SystemExit(f"missing environment: {', '.join(missing)}")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
WORK = DATA / "handoff" / os.environ["FS_RUN_ID"]
(WORK / "corpus").mkdir(parents=True, exist_ok=True)
(WORK / "annotated").mkdir(parents=True, exist_ok=True)
WORKERS = int(os.environ.get("FS_WORKERS", "4"))
print(f"work {WORK}\nworkers {WORKERS}")

In [ ]:
import os, json, hashlib
from pathlib import Path

try:
    import boto3
    from botocore.client import Config as BotoConfig
except ImportError as e:
    raise SystemExit("pip install boto3  # S3 handoff between the GPU box and the store box") from e

S3_ENDPOINT = os.environ.get("S3_ENDPOINT", "https://s3.wisefood-project.eu")
S3_BUCKET   = os.environ.get("S3_BUCKET", "foodscholar-graph-build")
RUN_ID      = os.environ["FS_RUN_ID"]          # same string on both machines

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    # MinIO speaks path-style; virtual-host style would resolve
    # <bucket>.s3.wisefood-project.eu, which has no DNS record.
    config=BotoConfig(signature_version="s3v4", s3={"addressing_style": "path"}),
)

def s3_key(*parts: str) -> str:
    return "/".join(["runs", RUN_ID, *parts])

def s3_exists(key: str) -> bool:
    try:
        s3.head_object(Bucket=S3_BUCKET, Key=key)
        return True
    except Exception:
        return False

def s3_put(local: Path, key: str) -> None:
    s3.upload_file(str(local), S3_BUCKET, key)

def s3_get(key: str, local: Path) -> Path:
    local.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(S3_BUCKET, key, str(local))
    return local

def s3_list(prefix: str) -> list[str]:
    keys, token = [], None
    while True:
        kw = {"Bucket": S3_BUCKET, "Prefix": prefix}
        if token:
            kw["ContinuationToken"] = token
        resp = s3.list_objects_v2(**kw)
        keys.extend(o["Key"] for o in resp.get("Contents", []))
        if not resp.get("IsTruncated"):
            return sorted(keys)
        token = resp["NextContinuationToken"]

print(f"s3  {S3_ENDPOINT}/{S3_BUCKET}")
print(f"run {RUN_ID}")

## 1. Pull the corpus and the ontology

In [ ]:
manifest = json.loads(s3_get(s3_key("manifest.json"), WORK / "manifest.json").read_text())
print(f"run {manifest['run_id']}: {manifest['chunks']} chunks in {len(manifest['shards'])} shards")

for f in ("foodon.owl", "foodon_cache.parquet", "foodon_cache.parquet.meta.json"):
    dest = DATA / f
    key = s3_key("ontology", f)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"  have {f}")
    elif s3_exists(key):
        t0 = time.perf_counter()
        s3_get(key, dest)
        print(f"  downloaded {f} ({dest.stat().st_size/1e6:.1f} MB, {time.perf_counter()-t0:.0f}s)")

shard_paths = []
for entry in manifest["shards"]:
    local = WORK / "corpus" / entry["shard"]
    if not local.exists():
        s3_get(entry["key"], local)
    shard_paths.append(local)
print(f"{len(shard_paths)} corpus shards local")

## 2. Configure — memory stores only

Nothing here writes to a database. `annotated_snapshot_path` is deliberately
unset: its skip-if-exists short-circuit applies to the whole phase, and this
notebook resumes per shard instead.

In [ ]:
from foodscholar import FoodScholar

CONFIG = {
    "corpus": {"chunks_path": str(WORK / "chunks.parquet"), "annotated_snapshot_path": None},
    "ontology": {"foodon_path": str(DATA / "foodon.owl"),
                 "cache_path": str(DATA / "foodon_cache.parquet"),
                 "include_imports": False},
    "annotate": {
        "ner": "gliner",
        "batch_size": int(os.environ.get("FS_ANNOTATE_BATCH", "16")),
        "linker": {
            "nel_backend": "hnsw",
            "nel_encoder": "biolord",
            # Pinned, not content-addressed, so every worker loads the one file
            # instead of each deriving a path and rebuilding the index.
            "nel_index_path": str(DATA / "foodon_hnsw_biolord.bin"),
            "nel_metadata_path": str(DATA / "foodon_hnsw_biolord.meta.json"),
        },
    },
    "storage": {"chunk_store": {"backend": "memory"},
                "graph_store": {"backend": "memory"},
                "card_store": {"backend": "memory"}},
}

fs = FoodScholar.from_config(CONFIG)
print("config hash:", fs.config_hash)

## 3. Warm the NEL index — once, before any worker starts

`HNSWNELIndex` builds from the ontology on first use and loads from disk
afterwards. Skip this and every worker encodes FoodOn simultaneously on the
first run.

In [ ]:
index_path = Path(CONFIG["annotate"]["linker"]["nel_index_path"])
t0 = time.perf_counter()
_ = fs.linker
print(f"NEL index ready in {time.perf_counter()-t0:.0f}s")
print(f"  {index_path} ({index_path.stat().st_size/1e6:.1f} MB)" if index_path.exists()
      else "  !! index missing — workers would each rebuild it")

## 4. Annotate, in parallel

Resumable twice over: a shard whose parquet is already on S3 is skipped
entirely, and one whose local parquet exists is not recomputed. Kill this cell
and re-run it — it picks up where it stopped.

In [ ]:
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

sys.path.insert(0, str(ROOT / "scripts" / "corpus"))
from annotate_shard import annotate_corpus_shard

# spawn, not fork: CUDA and fork deadlock. Harmless on a CPU-only box.
ctx = mp.get_context("spawn")

todo = []
for p in shard_paths:
    out = WORK / "annotated" / p.name.replace(".csv", ".parquet")
    if s3_exists(s3_key("annotated", out.name)):
        continue                       # already done, possibly by an earlier run
    todo.append((CONFIG, str(p), str(out)))

print(f"{len(todo)} shards to annotate ({len(shard_paths) - len(todo)} already on s3)")

totals = {"annotated": 0, "mentions": 0, "links": 0}
t0 = time.perf_counter()

if todo:
    with ProcessPoolExecutor(max_workers=WORKERS, mp_context=ctx) as pool:
        futures = {pool.submit(annotate_corpus_shard, job): job[2] for job in todo}
        for n, fut in enumerate(as_completed(futures), 1):
            try:
                r = fut.result()
            except Exception as exc:
                # One bad shard must not lose the other N-1. Its parquet is
                # never written, so re-running retries only that shard.
                print(f"  FAILED {Path(futures[fut]).name}: {exc}", flush=True)
                continue
            for k in totals:
                totals[k] += r.get(k, 0)
            elapsed = time.perf_counter() - t0
            rate = totals["annotated"] / elapsed if elapsed else 0
            left = (sum(s["rows"] for s in manifest["shards"]) - totals["annotated"]) / rate if rate else float("nan")
            print(f"  [{n}/{len(todo)}] {totals['annotated']} chunks | "
                  f"{rate:.1f}/s | eta {left/60:.0f} min", flush=True)

rate = totals["links"] / totals["mentions"] if totals["mentions"] else 0
print(f"\n{totals['annotated']} chunks in {(time.perf_counter()-t0)/60:.1f} min")
print(f"mentions {totals['mentions']}, links {totals['links']}, link rate {rate:.1%}")

## 5. Upload the results

The link rate above is the phase's quality signal. A run that annotated
everything and linked nothing is a broken NEL index, not a successful pass —
check it before uploading.

In [ ]:
uploaded = 0
for p in sorted((WORK / "annotated").glob("shard_*.parquet")):
    key = s3_key("annotated", p.name)
    if s3_exists(key):
        continue
    s3_put(p, key)
    uploaded += 1

done = s3_list(s3_key("annotated"))
print(f"uploaded {uploaded}; {len(done)} of {len(manifest['shards'])} shards on s3")

if len(done) < len(manifest["shards"]):
    print("\n!! incomplete — re-run the cell above to retry the missing shards")
else:
    print(f"\nComplete. Next: run 03_layers on the store box with FS_RUN_ID={RUN_ID}")